# Part 2 - MapReduce and Visualisation

## Setup

In [1]:
import os, subprocess, sys
from pathlib import Path

os.environ["JAVA_HOME"] = subprocess.run(
    ["/usr/libexec/java_home", "-v", "17"], capture_output=True, text=True).stdout.strip()
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

import pandas, pyspark
print("python ", sys.version.split()[0])
print("pyspark", pyspark.__version__)
print("pandas ", pandas.__version__)
print(subprocess.run(["java", "-version"], capture_output=True, text=True).stderr.splitlines()[0])
assert pyspark.__version__.startswith("4."), "Spark 4.x expected"

python  3.11.9
pyspark 4.2.0
pandas  2.3.3
openjdk version "17.0.20" 2026-07-21


In [2]:
import multiprocessing

REPO      = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").exists())
LOCAL     = REPO / "data"
LOCAL_RAW = LOCAL / "working"         
LOCAL_CUR = LOCAL / "curated"
LOCAL_OUT = REPO / "output"
EVENTS    = LOCAL / "spark-events"
DRIVE_OUT = REPO / "output"           
DRIVE_FIG = REPO / "figures"
for p in (LOCAL_CUR, LOCAL_OUT, EVENTS, DRIVE_OUT, DRIVE_FIG):
    p.mkdir(parents=True, exist_ok=True)

CORES      = multiprocessing.cpu_count()
DRIVER_MEM = "12g"                  
WINDOW     = [(y, m) for y in range(2022, 2026) for m in range(1, 13)]

MIN_NIGHT_TRIPS, MIN_PEAK_TRIPS = 100, 500
AIRPORTS = [132, 138]

print("repo", REPO, "| cores", CORES)

repo /Users/shreyabharat/Projects/MIT805-GroupProject | cores 8


In [3]:
import shutil

names   = [f"yellow_tripdata_{y}-{m:02d}.parquet" for y, m in WINDOW]
staged  = [str(LOCAL_RAW / n) for n in names if (LOCAL_RAW / n).exists()]
missing = [n for n in names if not (LOCAL_RAW / n).exists()]
nbytes  = sum(Path(p).stat().st_size for p in staged)

print(f"{len(staged)}/48 months present")
print(f"working set: {nbytes / 2**30:.2f} GiB")
print(f"missing: {missing or 'none'}")

free = shutil.disk_usage(LOCAL).free / 2**30
print(f"free disk: {free:.1f} GiB")
assert free > 25, "curated Parquet plus shuffle spill needs headroom"

48/48 months present
working set: 2.58 GiB
missing: none
free disk: 53.8 GiB


In [4]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName("MIT805-Part2-Congestion")
    .master(f"local[{CORES}]")
    .config("spark.driver.memory", DRIVER_MEM)
    .config("spark.sql.shuffle.partitions", CORES * 4)    
    .config("spark.sql.session.timeZone", "UTC")           
    .config("spark.sql.adaptive.enabled", "false")       
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", f"file://{EVENTS}")
    .config("spark.local.dir", str(LOCAL / "spark-tmp"))
    .getOrCreate())

spark.sparkContext.setLogLevel("WARN")
print(spark.version, "| cores:", CORES,
      "| shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
print("Spark UI:", spark.sparkContext.uiWebUrl)     # http://localhost:4040 while this session lives

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/24 21:18:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/24 21:18:02 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).


4.2.0 | cores: 8 | shuffle partitions: 32
Spark UI: http://localhost:4040


## 00: Data filtering and partitioning

In [5]:
from pyspark.sql import functions as F

def normalise(df):
    """TLC flips column capitalisation between eras; lowercase everything on read."""
    return df.toDF(*[c.lower() for c in df.columns])

raw = normalise(spark.read.parquet(*staged))

base = (raw
    .select(
        F.col("tpep_pickup_datetime").alias("pickup_ts"),
        F.col("tpep_dropoff_datetime").alias("dropoff_ts"),
        F.col("pulocationid").cast("int").alias("pu"),
        F.col("dolocationid").cast("int").alias("do"),
        F.col("trip_distance").cast("double").alias("miles"),
        F.col("fare_amount").cast("double").alias("fare"),
        F.col("tip_amount").cast("double").alias("tip"))
    .withColumn("duration_hr",
                F.expr("timestampdiff(SECOND, pickup_ts, dropoff_ts)") / 3600.0)
    .withColumn("speed_mph", F.expr("try_divide(miles, duration_hr)"))
    .withColumn("revenue", F.col("fare") + F.coalesce(F.col("tip"), F.lit(0.0)))
    .withColumn("hour", F.hour("pickup_ts"))
    .withColumn("daytype", F.when(F.dayofweek("pickup_ts").isin(1, 7), "weekend")
                            .otherwise("weekday")))

base.printSchema()

root
 |-- pickup_ts: timestamp_ntz (nullable = true)
 |-- dropoff_ts: timestamp_ntz (nullable = true)
 |-- pu: integer (nullable = true)
 |-- do: integer (nullable = true)
 |-- miles: double (nullable = true)
 |-- fare: double (nullable = true)
 |-- tip: double (nullable = true)
 |-- duration_hr: double (nullable = true)
 |-- speed_mph: double (nullable = true)
 |-- revenue: double (nullable = true)
 |-- hour: integer (nullable = true)
 |-- daytype: string (nullable = false)



In [6]:
CHECKS = {
    "in_window":   "year(pickup_ts) between 2022 and 2025",   # avoids NTZ vs zoned-literal mismatch
    "ordered":     "dropoff_ts > pickup_ts",
    "duration_pos": "duration_hr > 0",      # zero-duration trips exist: report the count
    "duration_ok": "duration_hr between 0.0166 and 3.0",      # 1 minute to 3 hours
    "distance_ok": "miles between 0.1 and 100",
    "speed_ok":    "speed_mph between 1 and 70",
    "money_ok":    "fare between 0.01 and 1000",
    "zones_ok":    "pu between 1 and 263 AND do between 1 and 263",   # 264/265 = Unknown / N.V.
}

audit = (base.select(
            F.count(F.lit(1)).alias("rows_read"),
            *[F.sum(F.when(F.expr(c), 1).otherwise(0)).alias(name) for name, c in CHECKS.items()])
         .collect()[0].asDict())
for k, v in audit.items():
    print(f"{k:14s} {v:>14,}  ({v / audit['rows_read']:.2%})")

keep  = " AND ".join(f"({c})" for c in CHECKS.values())
clean = base.filter(keep)

(clean
    .withColumn("year",  F.year("pickup_ts"))
    .withColumn("month", F.month("pickup_ts"))
    .repartition("year", "month")
    .write.mode("overwrite").partitionBy("year", "month")
    .parquet(str(LOCAL_CUR)))

curated = spark.read.parquet(str(LOCAL_CUR))
print("curated rows:", f"{curated.count():,}")

rows_read         167,858,646  (100.00%)
in_window         167,857,980  (100.00%)
ordered           167,251,228  (99.64%)
duration_pos      167,251,228  (99.64%)
duration_ok       165,152,624  (98.39%)
distance_ok       163,510,043  (97.41%)
speed_ok          163,174,278  (97.21%)
money_ok          163,575,252  (97.45%)
zones_ok          165,660,093  (98.69%)


curated rows: 156,964,574


In [7]:
(curated.groupBy("hour")
        .agg(F.count(F.lit(1)).alias("trips"),
             (F.sum("miles") / F.sum("duration_hr")).alias("speed_mph"),
             (F.sum("revenue") / F.sum("duration_hr")).alias("yield_per_hr"))
        .orderBy("hour").show(24))
# Expect: trips peak at hour 18, speed trough near hour 11, yield high in the early hours.

# Curated Parquet stays on disk — nothing to copy anywhere. Roughly how much did it cost?
print("curated size:",
      sum(f.stat().st_size for f in LOCAL_CUR.rglob("*.parquet")) / 2**30, "GiB")

+----+--------+------------------+------------------+
|hour|   trips|         speed_mph|      yield_per_hr|
+----+--------+------------------+------------------+
|   0| 4549628|16.471559982913565| 90.63224782569796|
|   1| 2988393| 16.18971217443901| 89.92943691957697|
|   2| 1962752|16.099872554901797| 89.37234344351398|
|   3| 1296741| 17.49705394187059|   92.663898642729|
|   4|  932992|21.154255343403666|103.42093191140475|
|   5| 1020115|22.592078858691963|105.20980437526539|
|   6| 2311243|18.349724841464123|  89.9262843221716|
|   7| 4424445|14.312760695974685| 80.22173312310528|
|   8| 6043802| 12.23982024846166| 76.23418540265315|
|   9| 6676952|11.797882743374117|  75.9012986824984|
|  10| 7190242|11.520921030710381| 75.02816907439885|
|  11| 7782712|11.073928518323033| 73.66683530799763|
|  12| 8446891|11.125623001148712| 73.53821900803514|
|  13| 8736449|11.300412891826108| 73.48813171191395|
|  14| 9355631|11.116455459201793| 71.83743953493754|
|  15| 9629118| 10.778702516

In [8]:
SLICE = LOCAL_CUR / "year=2024" / "month=6"

# Too big for git. Copy it to a Drive-synced folder, or zip it and send it once.
SHARE = Path.home() / "Library/CloudStorage/GoogleDrive-YOUR_EMAIL/My Drive/MIT805"
if SHARE.exists():
    subprocess.run(f"cp -r '{SLICE}' '{SHARE}/curated_slice_2024_06'", shell=True)
else:
    subprocess.run(f"cd '{LOCAL_CUR}' && zip -qr '{LOCAL_OUT}/curated_slice_2024_06.zip' "
                   f"'year=2024/month=6'", shell=True)
    print("zipped to", LOCAL_OUT / "curated_slice_2024_06.zip")

# The schema contract: Person 2 codes against this, so it must not change after 28 Sep.
schema_lines = [f"{f.name:14s} {f.dataType.simpleString()}" for f in curated.schema.fields]
(DRIVE_OUT / "curated_schema.txt").write_text(
    "MIT805 Part 2 curated schema - frozen 2026-09-25\n" + "\n".join(schema_lines))
print("\n".join(schema_lines))

zipped to /Users/shreyabharat/Projects/MIT805-GroupProject/output/curated_slice_2024_06.zip
pickup_ts      timestamp_ntz
dropoff_ts     timestamp_ntz
pu             int
do             int
miles          double
fare           double
tip            double
duration_hr    double
speed_mph      double
revenue        double
hour           int
daytype        string
year           int
month          int


## 01: Baseline

In [9]:
MIN_NIGHT_TRIPS = 100          # support filter — an analytical decision, justify it in the report

night = curated.filter("hour between 1 and 4 AND daytype = 'weekday'")

ff_route = (night
    .groupBy("pu", "do")
    .agg(F.sum("miles").alias("m"),
         F.sum("duration_hr").alias("h"),
         F.count(F.lit(1)).alias("n_night"))          # count(F.lit(1)), never count("*") — Spark 4 pivot gotcha
    .filter(F.col("n_night") >= MIN_NIGHT_TRIPS)
    .withColumn("ff_speed", F.col("m") / F.col("h"))
    .select("pu", "do", "ff_speed", "n_night")
    .cache())

ff_zone = (night
    .groupBy("pu")
    .agg(F.sum("miles").alias("m"),
         F.sum("duration_hr").alias("h"),
         F.sum("revenue").alias("r"),
         F.count(F.lit(1)).alias("n_night"))
    .withColumn("ff_speed_zone", F.col("m") / F.col("h"))
    .withColumn("ff_yield_zone", F.col("r") / F.col("h"))      # baseline $/vehicle-hour
    .select("pu", "ff_speed_zone", "ff_yield_zone", "n_night")
    .cache())

print("routes with a baseline:", f"{ff_route.count():,}", "| zones:", ff_zone.count())
ff_route.orderBy(F.desc("n_night")).show(5)

routes with a baseline: 3,599 | zones: 256
+---+---+------------------+-------+
| pu| do|          ff_speed|n_night|
+---+---+------------------+-------+
|246| 48|13.942572415634013|   9744|
| 48| 68|15.727113293581207|   8402|
|249| 79| 9.613682863798317|   8011|
| 48| 48|11.210423919460249|   7844|
| 79| 79|  9.93063004751946|   7130|
+---+---+------------------+-------+
only showing top 5 rows


## 02: RQ2 - Routes

## 03: RQ1 - Revenue

## 04: RDD Twin

## 05: Execution Evidence

## 06: Colab / Local Run 